# Quantum Password Cracker- using Grover's Algorithm

Grover's Algorithm is a powerful tool that has a quadratic speedup over classical algorithms. It is used widely in Unstructured search, pattern matching, as well as Quantum Cryptography- which is a fundamental area nowadays for secured communications. If a classical algorithm requires N number of steps, Grover's algorithm can do it in $\sqrt N$ steps.

In this, we attempt to create two different types of password crackers- a numeric and a binary.

Reminder: Restart the kernel everytime before running it again, in order to reset the circuit.

## Numerical Password Cracker

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from getpass import getpass
import math

password = getpass("Enter your numeric password: ")
n = bin(int(password))[2:]

if len(n) > 15:
    print(f"Error: {len(n)} qubits needed. Maximum supported is 15 qubits for simulation.")
    print("Real quantum hardware would be needed for larger passwords.")
elif len(n) > 10:
    print(f"Note: {len(n)} qubits needed. This may take a moment...")
N=2**len(n)

opt=math.floor((math.pi)/4*(math.sqrt(N)))

circuit=QuantumCircuit(len(n),len(n))

for i in range(len(n)):
    circuit.h(i)

for i in range(opt):
    #X wrapping of the qubits that are '0' in the target
    for j in range(len(n)):
        if n[j]=="0":
            circuit.x(j)
    #Applying CZ 
    if len(n)==2:
        circuit.cz(0,1)
    elif len(n)==3:
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
    else:
        circuit.h(len(n)-1)
        circuit.mcx(list(range(len(n)-1)), len(n)-1)
        circuit.h(len(n)-1)
    #Undo X Wrapping
    for j in range(len(n)):
        if n[j]=="0":
            circuit.x(j)
    # Diffuser
    for j in range(len(n)):
        circuit.h(j)
    for j in range(len(n)):
        circuit.x(j)
    if len(n)==2:
        circuit.cz(0,1)
    elif len(n)==3:
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
    else:
        circuit.h(len(n)-1)
        circuit.mcx(list(range(len(n)-1)), len(n)-1)
        circuit.h(len(n)-1)
    
    for j in range(len(n)):
        circuit.x(j)
    for j in range(len(n)):
        circuit.h(j)

for i in range(len(n)):
    circuit.measure(i,i)

simulator=AerSimulator()
job=simulator.run(circuit, shots=1000)
result=job.result()
counts=result.get_counts()

measured = max(counts, key=counts.get)[::-1]
cracked_password = int(measured, 2)
print(f"  Secret password:     {password}")
print(f"  Binary:              {n}")
print(f"  Cracked binary:      {measured}")
print(f"  Cracked password:    {cracked_password}")
print(f"  Match:               {'Yes' if cracked_password == int(password) else 'No'}")

Enter your numeric password:  ········


  Secret password:     100
  Binary:              1100100
  Cracked binary:      1100100
  Cracked password:    100
  Match:               Yes


## Binary Password Cracker

In [3]:
#Importing the header files

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from getpass import getpass
import math

n=getpass("Enter your password in binary:")

N=2**len(n)

opt=math.floor((math.pi)/4*(math.sqrt(N)))

circuit=QuantumCircuit(len(n),len(n))

for i in range(len(n)):
    circuit.h(i)

for i in range(opt):
    #X wrapping of the qubits that are '0' in the target
    for j in range(len(n)):
        if n[j]=="0":
            circuit.x(j)
    #Applying CZ 
    if len(n)==2:
        circuit.cz(0,1)
    elif len(n)==3:
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
    else:
        circuit.h(len(n)-1)
        circuit.mcx(list(range(len(n)-1)), len(n)-1)
        circuit.h(len(n)-1)
    #Undo X Wrapping
    for j in range(len(n)):
        if n[j]=="0":
            circuit.x(j)
    # Diffuser
    for j in range(len(n)):
        circuit.h(j)
    for j in range(len(n)):
        circuit.x(j)
    if len(n)==2:
        circuit.cz(0,1)
    elif len(n)==3:
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
    else:
        circuit.h(len(n)-1)
        circuit.mcx(list(range(len(n)-1)), len(n)-1)
        circuit.h(len(n)-1)
    
    for j in range(len(n)):
        circuit.x(j)
    for j in range(len(n)):
        circuit.h(j)

for i in range(len(n)):
    circuit.measure(i,i)

simulator=AerSimulator()
job=simulator.run(circuit, shots=1000)
result=job.result()
counts=result.get_counts()

measured = list(counts.keys())[0][::-1]
print("\n" + "=" * 40)
print("  QUANTUM PASSWORD CRACKER")
print("=" * 40)
print(f"  Secret password:     {n}")
print(f"  Cracked password:    {measured}")
print(f"  Match:               {'Yes' if measured == n else 'No'}")
print(f"  Classical checks:    up to {2**len(n)}")
print(f"  Grover iterations:   {opt}")
print(f"  Quantum speedup:     {2**len(n)}:{opt}")
print("=" * 40)

Enter your password in binary: ········



  QUANTUM PASSWORD CRACKER
  Secret password:     110101110
  Cracked password:    110101110
  Match:               Yes
  Classical checks:    up to 512
  Grover iterations:   17
  Quantum speedup:     512:17
